# 05 — Rank-difference deconvolution

Join Yan 2020 DMS with **ESM-2 650M** and SPURS. The KDE model flags mutants
whose evolutionary rank is much worse than their stability rank (`neg_spurs`
or DMS `stability`).

If an `EVmutation` column is present (EVcouplings webserver), it is deconvolved
as well; ESM-2 650M is sufficient to run the example.


## Load inputs


In [ ]:
from IPython.display import display
import pandas as pd

from dualde.deconvolution import deconvolve_evo_vs_stability, merge_evo_spurs_dms
from dualde.paths import example_dir

EXAMPLE = example_dir("NUDT15_HUMAN")
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

evo_path = OUT / "evolutionary.csv" if (OUT / "evolutionary.csv").is_file() else EXAMPLE / "scores" / "evolutionary.csv"
spurs_path = OUT / "5lpg_spurs.csv" if (OUT / "5lpg_spurs.csv").is_file() else EXAMPLE / "scores" / "5lpg_spurs.csv"

evo = pd.read_csv(evo_path)
spurs = pd.read_csv(spurs_path)
activity = pd.read_csv(EXAMPLE / "dms" / "yan2020_activity.csv")
stability = pd.read_csv(EXAMPLE / "dms" / "yan2020_stability.csv")

print("evo", evo.shape, "from", evo_path.name)
print("spurs", spurs.shape)
print("activity", activity.shape, "stability", stability.shape)
display(activity.head(3))
display(stability.head(3))


## Merge


In [ ]:
merged = merge_evo_spurs_dms(evo, spurs, activity, stability)
print("merged", merged.shape)
print("columns:", list(merged.columns))
print("missingness (key cols):")
key = [c for c in ["mutant", "activity", "stability", "spurs", "neg_spurs", "ESM2_650M", "EVmutation"] if c in merged.columns]
display(merged[key].isna().mean().to_frame("frac_na"))
merged.head()


## KDE models (ESM2 vs stability proxies)


In [ ]:
evo_cols = [c for c in ("ESM2_650M", "EVmutation") if c in merged.columns]
print("deconvolving:", evo_cols, "vs neg_spurs and DMS stability")
merged = deconvolve_evo_vs_stability(merged, evo_cols=evo_cols, stability_cols=("neg_spurs", "stability"))
model_cols = [c for c in merged.columns if c.endswith("_model")]
print("model columns:", model_cols)
merged.to_csv(OUT / "deconvolution.csv", index=False)
print("wrote", OUT / "deconvolution.csv")


Higher `*_model` = more likely an evolutionary outlier given the stability score (putative catalytic / functional effect).


In [ ]:
score_col = "ESM2_650M_neg_spurs_model" if "ESM2_650M_neg_spurs_model" in merged.columns else model_cols[0]
print("inspecting", score_col)
print(merged[score_col].describe())
top = merged.nlargest(8, score_col)[["mutant", "target_pos", "activity", "stability", "neg_spurs", score_col]]
display(top)
